In [2]:
# Importar librerias

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_wine

import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
wine = load_wine(as_frame = True)

In [4]:
print(wine.DESCR)

.. _wine_dataset:

Wine recognition dataset
------------------------

**Data Set Characteristics:**

:Number of Instances: 178
:Number of Attributes: 13 numeric, predictive attributes and the class
:Attribute Information:
    - Alcohol
    - Malic acid
    - Ash
    - Alcalinity of ash
    - Magnesium
    - Total phenols
    - Flavanoids
    - Nonflavanoid phenols
    - Proanthocyanins
    - Color intensity
    - Hue
    - OD280/OD315 of diluted wines
    - Proline
    - class:
        - class_0
        - class_1
        - class_2

:Summary Statistics:

============================= ==== ===== ======= =====
                                Min   Max   Mean     SD
============================= ==== ===== ======= =====
Alcohol:                      11.0  14.8    13.0   0.8
Malic Acid:                   0.74  5.80    2.34  1.12
Ash:                          1.36  3.23    2.36  0.27
Alcalinity of Ash:            10.6  30.0    19.5   3.3
Magnesium:                    70.0 162.0    99.7  14.3

In [5]:
dfWine = wine.frame

In [6]:
dfWine.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0


## Exploración del conjunto de datos

In [7]:
dfWine.shape

(178, 14)

In [8]:
dfWine.columns

Index(['alcohol', 'malic_acid', 'ash', 'alcalinity_of_ash', 'magnesium',
       'total_phenols', 'flavanoids', 'nonflavanoid_phenols',
       'proanthocyanins', 'color_intensity', 'hue',
       'od280/od315_of_diluted_wines', 'proline', 'target'],
      dtype='object')

In [9]:
wine.target_names

array(['class_0', 'class_1', 'class_2'], dtype='<U7')

## Renombrar la columna target

In [10]:
dfWine.rename(columns = {"target": "especie"}, inplace = True)

In [11]:
dfWine["especie"] = dfWine["especie"].map({
    0: "class_0",
    1: "class_1",
    2: "class_2"
})

In [15]:
dfWine.head(100)

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,especie
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,class_0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,class_0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,class_0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,class_0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,class_0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,12.47,1.52,2.20,19.0,162.0,2.50,2.27,0.32,3.28,2.60,1.16,2.63,937.0,class_1
96,11.81,2.12,2.74,21.5,134.0,1.60,0.99,0.14,1.56,2.50,0.95,2.26,625.0,class_1
97,12.29,1.41,1.98,16.0,85.0,2.55,2.50,0.29,1.77,2.90,1.23,2.74,428.0,class_1
98,12.37,1.07,2.10,18.5,88.0,3.52,3.75,0.24,1.95,4.50,1.04,2.77,660.0,class_1


In [16]:
dfWine.tail()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,especie
173,13.71,5.65,2.45,20.5,95.0,1.68,0.61,0.52,1.06,7.7,0.64,1.74,740.0,class_2
174,13.40,3.91,2.48,23.0,102.0,1.80,0.75,0.43,1.41,7.3,0.70,1.56,750.0,class_2
175,13.27,4.28,2.26,20.0,120.0,1.59,0.69,0.43,1.35,10.2,0.59,1.56,835.0,class_2
176,13.17,2.59,2.37,20.0,120.0,1.65,0.68,0.53,1.46,9.3,0.60,1.62,840.0,class_2
177,14.13,4.10,2.74,24.5,96.0,2.05,0.76,0.56,1.35,9.2,0.61,1.60,560.0,class_2


## Identificar variables predictoras y variables objetivo

In [17]:
# Variables predictoras

X = dfWine.drop(columns = ["especie"])

In [18]:
print("Variables predictoras")
print(X.columns)

Variables predictoras
Index(['alcohol', 'malic_acid', 'ash', 'alcalinity_of_ash', 'magnesium',
       'total_phenols', 'flavanoids', 'nonflavanoid_phenols',
       'proanthocyanins', 'color_intensity', 'hue',
       'od280/od315_of_diluted_wines', 'proline'],
      dtype='object')


In [19]:
# Variable objetivo

y = dfWine["especie"]

In [20]:
print("Variables predictoras")
print(y.name)

Variables predictoras
especie


## Evaluar los datos en prueba y entrenamiento
Vamos a evaluar los datos 80% para entrenamiento y 20% para prueba

In [21]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify = y
)

In [22]:
print("X_train", X_train.shape)
print("X_test", X_test.shape)
print("y_train", y_train.shape)
print("y_test", y_test.shape)

X_train (142, 13)
X_test (36, 13)
y_train (142,)
y_test (36,)


## Escalado de variables

In [24]:
scaler = StandardScaler()

In [25]:
# Escalamos las variables

X_train_escalado = scaler.fit_transform(
    X_train
)
X_test_escalado = scaler.transform(
    X_test
)

In [26]:
print("Sin escalar")
print(X_train.head())
print("Escalado")
print(X_train_escalado[:5])

Sin escalar
     alcohol  malic_acid   ash  alcalinity_of_ash  magnesium  total_phenols  \
36     13.28        1.64  2.84               15.5      110.0           2.60   
30     13.73        1.50  2.70               22.5      101.0           3.00   
26     13.39        1.77  2.62               16.1       93.0           2.85   
12     13.75        1.73  2.41               16.0       89.0           2.60   
148    13.32        3.24  2.38               21.5       92.0           1.93   

     flavanoids  nonflavanoid_phenols  proanthocyanins  color_intensity   hue  \
36         2.68                  0.34             1.36             4.60  1.09   
30         3.25                  0.29             2.38             5.70  1.19   
26         2.94                  0.34             1.45             4.80  0.92   
12         2.76                  0.29             1.81             5.60  1.15   
148        0.76                  0.45             1.25             8.42  0.55   

     od280/od315_of_dilute

## Entrenamiento del modelo KNN
En esta prueba utilizaremos k = 3

In [27]:
modelo = KNeighborsClassifier(n_neighbors = 3)

In [28]:
modelo.fit(
    X_train_escalado, y_train
)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",3
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.Doesn't affect :meth:`fit` method.",None
Name,Type,Value
"classes_ classes_: array of shape (n_classes,)Class labels known to the classifier","ndarray[object](3,)","['class_0','class_1','class_2']"
"effective_metric_ effective_metric_: str or callbleThe distance metric used. It will be same as the `metric` parameteror a synonym of it, e.g. 'euclidean' if the `metric` parameter set to'minkowski' and `p` parameter set to 2.",str,'eu...an'


## Realizando predicciones, con las variables de prueba

In [29]:
prediccion = modelo.predict(X_test_escalado)

In [30]:
print(prediccion[:10])

['class_0' 'class_2' 'class_0' 'class_0' 'class_1' 'class_0' 'class_0'
 'class_1' 'class_1' 'class_2']


In [31]:
## Comparación entre valores reales y valores predecidos
comparacion = pd.DataFrame({
    "especie real": y_test.values,
    "especie predicha": prediccion
})

In [32]:
comparacion.head(40)

,especie real,especie predicha
0,class_0,class_0
1,class_2,class_2
2,class_0,class_0
3,class_1,class_0
4,class_1,class_1
5,class_0,class_0
6,class_0,class_0
7,class_1,class_1
8,class_1,class_1
9,class_2,class_2


## Evaluación del modelo Accurancy
Representa la cantdad de predicciones correctas que dio el modelo.

In [33]:
accuracy = accuracy_score(y_test, prediccion)
print(f"accuracy/exactitud:  {accuracy: .4f}")

accuracy/exactitud:   0.9722


In [34]:
from sklearn.metrics import confusion_matrix

# Matriz de confusión
cm = confusion_matrix(y_test, prediccion)

In [35]:
cm_df = pd.DataFrame(
    cm,
    index = modelo.classes_,
    columns = modelo.classes_
)
cm_df

,class_0,class_1,class_2
class_0,12,0,0
class_1,1,13,0
class_2,0,0,10


## Clasification report
Proporciona metricas que permiten evaluar el desempeño de cada clase: Precision, recall, f1-score, support

In [36]:
from sklearn.metrics import classification_report

print(classification_report(y_test, prediccion ))

              precision    recall  f1-score   support

     class_0       0.92      1.00      0.96        12
     class_1       1.00      0.93      0.96        14
     class_2       1.00      1.00      1.00        10

    accuracy                           0.97        36
   macro avg       0.97      0.98      0.97        36
weighted avg       0.97      0.97      0.97        36

